# Mastering Pandas for AI Engineering

## Learning Goal
- **What this notebook teaches**: Pandas DataFrames from loading to cleaning to aggregation and export — with a focus on the patterns that appear in real AI dataset pipelines.
- **Background & links**: Pandas docs: https://pandas.pydata.org/docs/ | User guide: https://pandas.pydata.org/docs/user_guide/index.html
- **Why it matters in AI engineering**: Every NLP dataset, evaluation spreadsheet, annotation file, and benchmark result lands in a CSV or JSONL. Pandas is the universal tool for loading, cleaning, filtering, and preparing that data before it reaches a model. Sloppy Pandas skills = hours lost on data bugs.

## Mental Model

```
              ┌─────────────────────────────────────────────────────────┐
              │                  Pandas DataFrame                       │
              │                                                         │
  CSV  ──────▶│  Index │  col_A (object)  col_B (int64)  col_C (f32) │──▶  NumPy
  JSON ──────▶│  ──────│─────────────────────────────────────────────  │──▶  Dict
  JSONL──────▶│    0   │  "Alice"          85             0.91         │──▶  JSON
  Parquet────▶│    1   │  "Bob"            92             0.88         │──▶  Parquet
              │    2   │  "Carol"          78             0.95         │
              │                                                         │
              │  ← each column is a named Pandas Series (= NumPy array │
              │    + labels + dtype + NaN support)                      │
              └─────────────────────────────────────────────────────────┘
```

A DataFrame is **a dict of named Series**, each Series backed by a NumPy array.
Column operations are vectorized. Row-wise Python loops are almost always wrong.

## Background Explanation

### Essential operations at a glance

| Task | Code |
|------|------|
| Load CSV | `pd.read_csv("file.csv")` |
| Load JSON / JSONL | `pd.read_json("f.jsonl", lines=True)` |
| Quick peek | `df.head()`, `df.info()`, `df.describe()` |
| Select column | `df["col"]` → Series |
| Select columns | `df[["a","b"]]` → DataFrame |
| Filter rows | `df[df["score"] > 0.8]` |
| Multi-filter | `df[(cond1) & (cond2)]` |
| New column | `df["new"] = expression` |
| Apply function | `df["col"].apply(fn)` |
| Group & aggregate | `df.groupby("col").agg({"a": "mean", "b": "sum"})` |
| Sort | `df.sort_values("col", ascending=False)` |
| Drop duplicates | `df.drop_duplicates(subset=["col"])` |
| Handle missing | `df.dropna()`, `df.fillna(value)` |
| Rename columns | `df.rename(columns={"old": "new"})` |
| Export | `df.to_csv()`, `df.to_json()`, `df.to_parquet()` |

### Key concepts
- **Index** — the row labels (integer by default, but can be any hashable)
- **Series** — 1D labeled array, like a single column
- **NaN** — `float('nan')` in Python, propagates through operations
- **`loc` vs `iloc`** — `loc` is label-based, `iloc` is position-based

In [ ]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)

## Minimal Working Example

In [ ]:
# Build a small evaluation DataFrame
data = {
    "id":      [1, 2, 3, 4, 5],
    "model":   ["claude-sonnet-4-5", "gpt-4o", "claude-sonnet-4-5", "arabic-llm", "gpt-4o"],
    "score":   [0.91, 0.88, 0.75, 0.94, 0.82],
    "passed":  [True, True, False, True, True],
    "tokens":  [405, 340, 610, 280, 520],
}
df = pd.DataFrame(data)
print(df)
print("\nShape:", df.shape)
print("Dtypes:\n", df.dtypes)

In [ ]:
# Quick exploration
print(df.describe())
print("\nPassed rate:", df["passed"].mean().round(2))
print("\nMean score per model:")
print(df.groupby("model")["score"].mean().sort_values(ascending=False))

## Exercise 1 — Clean and Analyse an Arabic NLP Dataset

**Goal**: Load a messy dataset, clean it, compute statistics, and export a cleaned version.

**Real context**: In Arabic NLP projects, annotated datasets often arrive with encoding issues, duplicate records, missing labels, and inconsistent formatting.

In [ ]:
# Simulate a messy Arabic NLP annotation file
import io

messy_csv = """id,text,label,confidence,annotator
1,مرحباً بالعالم,greeting,0.95,ann_1
2,كيف حالك؟,question,0.88,ann_2
3,مرحباً بالعالم,greeting,0.95,ann_1
4,أنا لا أعرف,unknown,,ann_1
5,شكراً جزيلاً,thanks,0.91,ann_2
6,ما هو الذكاء الاصطناعي؟,question,0.77,ann_3
7,,question,0.60,ann_1
8,هذا رائع جداً,praise,0.85,ann_2
9,لا أفهم هذا,confusion,None,ann_3
10,مرحباً بالعالم,greeting,0.95,ann_1
"""

df_raw = pd.read_csv(io.StringIO(messy_csv))
print("Raw DataFrame:")
print(df_raw)
print("\nInfo:")
df_raw.info()

In [ ]:
def clean_arabic_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset with the following steps:
    1. Drop rows where 'text' is missing (NaN)
    2. Drop exact duplicate rows (same id, text, label all matching)
    3. Fix 'confidence': convert 'None' strings and NaN to 0.0, cast to float
    4. Add a new column 'text_length' = number of characters in 'text'
    5. Filter to keep only rows where confidence >= 0.75 OR confidence == 0.0 (unknown)
    6. Reset the index (drop=True)
    Return the cleaned DataFrame.
    """
    df = df.copy()   # never mutate the original

    # TODO 1: Drop rows with missing 'text'

    # TODO 2: Drop duplicate rows based on ['text', 'label'] (keep first)

    # TODO 3: Fix confidence column
    # Hint: df["confidence"] = df["confidence"].replace("None", np.nan).astype(float).fillna(0.0)

    # TODO 4: Add text_length column

    # TODO 5: Filter rows

    # TODO 6: Reset index

    return df

df_clean = clean_arabic_dataset(df_raw)
print(f"Rows before: {len(df_raw)} → after: {len(df_clean)}")
print(df_clean)

**Questions to think about**
1. Why do we `df = df.copy()` at the start instead of mutating `df` directly?
2. What is the difference between `dropna()` and `fillna()`? When would you use each?
3. Why is `df[(cond1) & (cond2)]` instead of `df[cond1 and cond2]`?

## Solution 1

In [ ]:
def clean_arabic_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1 — drop rows with missing text
    df = df.dropna(subset=["text"])

    # 2 — drop duplicates on text + label (keep first occurrence)
    df = df.drop_duplicates(subset=["text", "label"])

    # 3 — fix confidence: replace string "None", cast, fill NaN with 0.0
    df["confidence"] = (df["confidence"]
                        .replace("None", np.nan)
                        .astype(float)
                        .fillna(0.0))

    # 4 — add text length
    df["text_length"] = df["text"].str.len()

    # 5 — filter: keep high-confidence OR confidence == 0.0 (needs review)
    df = df[(df["confidence"] >= 0.75) | (df["confidence"] == 0.0)]

    # 6 — reset index
    df = df.reset_index(drop=True)
    return df

df_clean = clean_arabic_dataset(df_raw)
print(f"Rows: {len(df_raw)} → {len(df_clean)}")
print(df_clean.to_string())

## Exercise 2 — Evaluation Metrics by Model and Label

**Goal**: Compute a multi-level aggregation report — mean score, pass rate, and token usage — grouped by model and label type. Then export to JSONL.

**Real context**: After running evals, you need to slice results by model and by task type to understand where each model excels or fails.

In [ ]:
np.random.seed(42)
n = 40
models  = np.random.choice(["claude-sonnet-4-5","gpt-4o","arabic-llm"], n)
labels  = np.random.choice(["summarization","translation","qa","classification"], n)
scores  = np.round(np.random.uniform(0.6, 1.0, n), 3)
tokens  = np.random.randint(100, 800, n)
passed  = scores >= 0.80

eval_df = pd.DataFrame({
    "model": models, "task": labels,
    "score": scores, "tokens": tokens, "passed": passed
})

def compute_eval_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group by (model, task) and compute:
      - n_runs       : count of rows
      - mean_score   : mean of score (round to 3 dp)
      - pass_rate    : mean of passed (round to 2 dp)
      - avg_tokens   : mean of tokens (round to 0 dp, as int)
    Return a DataFrame sorted by mean_score descending, index reset.
    """
    # TODO: implement using groupby + agg
    pass

report = compute_eval_report(eval_df)
print(report.to_string())

In [ ]:
# After completing the function above, export to JSONL
# TODO: export `report` to a JSONL string using df.to_json(orient="records", lines=True)
# Then verify you can read it back with pd.read_json(..., lines=True)
pass

## Solution 2

In [ ]:
def compute_eval_report(df: pd.DataFrame) -> pd.DataFrame:
    report = (df.groupby(["model", "task"])
                .agg(
                    n_runs      =("score",  "count"),
                    mean_score  =("score",  "mean"),
                    pass_rate   =("passed", "mean"),
                    avg_tokens  =("tokens", "mean"),
                )
                .round({"mean_score": 3, "pass_rate": 2, "avg_tokens": 0})
                .sort_values("mean_score", ascending=False)
                .reset_index())
    report["avg_tokens"] = report["avg_tokens"].astype(int)
    return report

report = compute_eval_report(eval_df)
print(report.to_string())

# Export to JSONL and read back
import io
jsonl_str = report.to_json(orient="records", lines=True)
report_back = pd.read_json(io.StringIO(jsonl_str), lines=True)
print("\nRound-trip OK:", report_back.shape == report.shape)

## Common Mistakes

### Mistake 1 — Chained indexing (silent no-op on assignment)

In [ ]:
df_test = pd.DataFrame({"a": [1,2,3], "b": [4,5,6]})

# WRONG — may silently fail to modify original
try:
    df_test[df_test["a"] > 1]["b"] = 99   # SettingWithCopyWarning
except Exception as e:
    print(f"Error: {e}")

# CORRECT — use .loc with boolean mask
df_test.loc[df_test["a"] > 1, "b"] = 99
print(df_test)

### Mistake 2 — `apply` in a loop instead of vectorized operations

In [ ]:
import time

df_big = pd.DataFrame({"text": ["hello world"] * 10_000})

# SLOW — Python loop via apply
t0 = time.perf_counter()
df_big["length_slow"] = df_big["text"].apply(len)
t1 = time.perf_counter()

# FAST — vectorized string operation
t2 = time.perf_counter()
df_big["length_fast"] = df_big["text"].str.len()
t3 = time.perf_counter()

print(f"apply  : {(t1-t0)*1000:.1f} ms")
print(f"str.len: {(t3-t2)*1000:.1f} ms")
print("Results equal:", (df_big["length_slow"] == df_big["length_fast"]).all())

### Mistake 3 — Forgetting `reset_index` after groupby

In [ ]:
df_small = pd.DataFrame({"model":["a","a","b","b"], "score":[0.8,0.9,0.7,0.85]})

grouped = df_small.groupby("model")["score"].mean()
print("Without reset_index:\n", grouped)
print("Type:", type(grouped))  # Series with 'model' as INDEX

grouped_df = df_small.groupby("model")["score"].mean().reset_index()
print("\nWith reset_index:\n", grouped_df)
print("Now 'model' is a normal column, not the index")

## Real AI Engineering Usage

```python
import pandas as pd

# 1. Load and inspect an Arabic NLP benchmark dataset
df = pd.read_json("arabic_bench.jsonl", lines=True)
print(df["label"].value_counts())   # check class balance

# 2. Build a training manifest (text + label pairs ready for tokeniser)
manifest = df[["text", "label"]].dropna().reset_index(drop=True)
manifest.to_json("train_manifest.jsonl", orient="records", lines=True, force_ascii=False)

# 3. Evaluation: merge model predictions with ground truth
preds  = pd.read_json("predictions.jsonl", lines=True)
merged = ground_truth.merge(preds, on="id", suffixes=("_gt", "_pred"))
merged["correct"] = merged["label_gt"] == merged["label_pred"]
accuracy = merged["correct"].mean()

# 4. Cost tracking — compute token spend per experiment
logs = pd.read_json("api_log.jsonl", lines=True)
logs["cost"] = (logs["input_tokens"] / 1e6 * 3.00 +
                logs["output_tokens"] / 1e6 * 15.00)
print(logs.groupby("experiment_id")["cost"].sum().sort_values(ascending=False))

# 5. Export to Parquet for fast re-loading in training loop
df_clean.to_parquet("dataset_clean.parquet", index=False)
df_fast  = pd.read_parquet("dataset_clean.parquet")   # 10× faster than CSV
```

## Final Mini Exercise — Dataset Quality Report

Given a raw DataFrame, generate a quality report as a new DataFrame with one row per column containing: column name, dtype, null count, null percentage, and number of unique values.

In [ ]:
raw = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6],
    "text":       ["hello", None, "world", "hello", None, "!"],
    "label":      ["pos", "neg", "pos", "pos", "neg", None],
    "confidence": [0.9, 0.8, None, 0.7, 0.85, 0.92],
    "model":      ["A", "A", "B", "B", "A", "B"],
})

def dataset_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
      column | dtype | null_count | null_pct | n_unique
    One row per column in df.
    """
    # TODO: implement
    pass

print(dataset_quality_report(raw).to_string(index=False))

## Key Takeaways

- A DataFrame is a **dict of named NumPy arrays** with row/column labels and NaN support.
- Always `df = df.copy()` before mutating inside a function — never modify the caller's data.
- Use **vectorized** operations (`str.len()`, `str.contains()`, arithmetic) over `.apply()` wherever possible.
- `groupby` + `agg` is the idiomatic way to compute per-group statistics — prefer it over loops.
- `loc[mask, col] = value` is the safe way to conditionally set values — avoid chained indexing.
- `read_json(lines=True)` and `to_json(lines=True)` are the JSONL interface — critical for large AI logs.
- Export to **Parquet** for large datasets that will be reloaded frequently — 10× faster than CSV.
- `ensure_ascii=False` in `to_json()` is required to preserve Arabic text correctly.